FASE 2 — Data Understanding: ILMM 2020

## Construcción y validación de ILMM 2020

### Proyecto
Modelo econométrico de atractividad comercial municipal en México.

### Objetivo de esta etapa

Explorar y validar la información de los Indicadores Laborales para los Municipios de México (ILMM) 2020.

La variable laboral que se construirá para el modelo será:

`TASA_INFORMALIDAD_2020`

definida como:

INFORMALES / OCUPADOS × 100

Esta variable será posteriormente utilizada como:

`X3 = Tasa de informalidad laboral municipal`

Antes de construirla se verificará:

- estructura interna del paquete ILMM;
- archivos disponibles;
- cobertura municipal;
- variables laborales disponibles;
- definición oficial de las variables;
- claves municipales;
- valores faltantes;
- duplicados;
- consistencia entre ocupados, formales e informales.

En esta fase todavía no se integrará ILMM con las demás fuentes ni se realizarán regresiones.

In [1]:
############  2.2 Librerías y rutas

from pathlib import Path
import zipfile
from io import BytesIO

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
PROJECT_ROOT = Path.cwd().parent

ILMM_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "ilmm"
)

archivo_ilmm = (
    ILMM_DIR
    / "ilmm_2020_bd_xlsx.zip"
)

print("Ruta del proyecto:")
print(PROJECT_ROOT)

print("\nRuta ILMM:")
print(archivo_ilmm)

print("\n¿Existe el archivo?:", archivo_ilmm.exists())

Ruta del proyecto:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico

Ruta ILMM:
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\raw\ilmm\ilmm_2020_bd_xlsx.zip

¿Existe el archivo?: True




### 2.3 Inspección de la estructura del paquete ILMM 2020

Antes de cargar los datos se revisa el contenido del paquete oficial.

El objetivo es identificar:

- archivos de bases de datos;
- descriptor de archivos;
- documentación y algoritmos incluidos en la fuente.

In [3]:
with zipfile.ZipFile(archivo_ilmm, "r") as z:

    contenido_ilmm = z.namelist()

    print(
        f"Número de archivos encontrados: "
        f"{len(contenido_ilmm)}\n"
    )

    for nombre in contenido_ilmm:
        print(nombre)

Número de archivos encontrados: 3

base_datos.zip
descripcion_archivos_fd.xlsx
enoeeap_2020_algoritmo.zip


In [4]:
#######  2.4 Inspeccionar base_datos.zip

with zipfile.ZipFile(archivo_ilmm, "r") as z:

    base_datos_bytes = z.read("base_datos.zip")

with zipfile.ZipFile(
    BytesIO(base_datos_bytes),
    "r"
) as z_base:

    contenido_base = z_base.namelist()

    print(
        f"Número de archivos dentro de base_datos.zip: "
        f"{len(contenido_base)}\n"
    )

    for nombre in contenido_base:
        print(nombre)

Número de archivos dentro de base_datos.zip: 13

EAP_ESTIMACIONES_PEA_OCU_INF_2020_T1.xlsx
ENOE_ARCHIVO_MAESTRO_2020_T1.xlsx
ENOE_INF_2020.xlsx
ENOE_PEA_OCU_2020.xlsx
ENOE_XENTI_2020_T1.csv
MUNI.CPG
muni.DBF
MUNI.prj
MUNI.sbn
MUNI.sbx
MUNI.shp
MUNI.shx
MUNI.xml


### 2.5 Inspección de la base de estimaciones municipales

Se revisa el archivo de estimaciones en áreas pequeñas que contiene las cifras laborales municipales necesarias para el proyecto.

Las variables de principal interés son:

- `LLAVE`: clave municipal;
- `MUNICIPIO`: nombre del municipio;
- `OCUPADOS`: población ocupada estimada;
- `INFORMALES`: población ocupada informal estimada;
- `FORMALES`: población ocupada formal estimada;
- `EstisiINF`: estimación publicada de la tasa de informalidad.

Todavía no se construye la variable final.

In [5]:
with zipfile.ZipFile(
    BytesIO(base_datos_bytes),
    "r"
) as z_base:

    archivo_estimaciones = (
        "EAP_ESTIMACIONES_PEA_OCU_INF_2020_T1.xlsx"
    )

    estimaciones_bytes = z_base.read(
        archivo_estimaciones
    )

excel_estimaciones = pd.ExcelFile(
    BytesIO(estimaciones_bytes)
)

print("Hojas disponibles:")
print(excel_estimaciones.sheet_names)

Hojas disponibles:
['Sheet 1']


In [6]:
muestra_ilmm = pd.read_excel(
    BytesIO(estimaciones_bytes),
    sheet_name=excel_estimaciones.sheet_names[0],
    nrows=10
)

print("Dimensiones de la muestra:")
print(muestra_ilmm.shape)

print("\nColumnas:")
print(muestra_ilmm.columns.tolist())

display(muestra_ilmm)

Dimensiones de la muestra:
(10, 15)

Columnas:
['LLAVE', 'MUNICIPIO', 'T15ymas', 'EstisiPEA', 'RECMPEA', 'EstisiOCU', 'RECMOCU', 'EstisiINF', 'RECMINF', 'PEA', 'NO_PEA', 'OCUPADOS', 'DESOCUPADOS', 'FORMALES', 'INFORMALES']


,LLAVE,MUNICIPIO,T15ymas,EstisiPEA,RECMPEA,EstisiOCU,RECMOCU,EstisiINF,RECMINF,PEA,NO_PEA,OCUPADOS,DESOCUPADOS,FORMALES,INFORMALES
0,1001,Aguascalientes,706146.451442,0.609054,0.006203,0.591558,0.005591,0.251977,0.026122,430081.451542,276064.999900,417726.792991,12354.658551,239794.282877,177932.510114
1,1002,Asientos,35175.941192,0.594906,0.010524,0.586544,0.003395,0.426814,0.014916,20926.370626,14249.570566,20632.222968,294.147657,5618.626438,15013.596531
2,1003,Calvillo,41447.094488,0.586366,0.011668,0.570542,0.005821,0.358614,0.012411,24303.158228,17143.936260,23647.320806,655.837423,8783.802127,14863.518679
3,1004,Cosío,11793.687552,0.596951,0.013634,0.581910,0.007064,0.323247,0.016689,7040.251063,4753.436489,6862.859091,177.391972,3050.583901,3812.275190
4,1005,Jesús María,91295.008850,0.617918,0.010040,0.601883,0.005953,0.298674,0.013192,56412.804983,34882.203867,54948.951243,1463.853740,27681.534445,27267.416798
5,1006,Pabellón de Arteaga,33427.873425,0.603009,0.010009,0.600903,0.003805,0.346279,0.021519,20157.299652,13270.573772,20086.925736,70.373917,8511.566423,11575.359312
6,1007,Rincón de Romos,39439.048672,0.593462,0.009932,0.594742,0.003798,0.355238,0.023308,23405.589527,16033.459146,23405.589527,0.000000,9395.324975,14010.264552
7,1008,San José de Gracia,6507.328147,0.577289,0.012292,0.563358,0.006399,0.394814,0.015394,3756.609718,2750.718429,3665.957417,90.652302,1096.775639,2569.181777
8,1009,Tepezalá,15546.257951,0.593880,0.012624,0.579751,0.007052,0.377612,0.016401,9232.619302,6313.638648,9012.958356,219.660946,3142.508914,5870.449442
9,1010,El Llano,14587.250897,0.602631,0.010501,0.590712,0.004507,0.361336,0.013949,8790.727742,5796.523155,8616.867297,173.860445,3345.963775,5270.903522


In [7]:
######   2.6 Validar las variables que necesitamos

variables_ilmm_requeridas = [
    "LLAVE",
    "MUNICIPIO",
    "T15ymas",
    "PEA",
    "OCUPADOS",
    "FORMALES",
    "INFORMALES",
    "EstisiINF"
]

validacion_variables_ilmm = pd.DataFrame({
    "variable": variables_ilmm_requeridas,
    "disponible": [
        variable in muestra_ilmm.columns
        for variable in variables_ilmm_requeridas
    ]
})

display(validacion_variables_ilmm)

variables_faltantes_ilmm = [
    variable
    for variable in variables_ilmm_requeridas
    if variable not in muestra_ilmm.columns
]

print(
    "Variables faltantes:",
    variables_faltantes_ilmm
)

,variable,disponible
0,LLAVE,True
1,MUNICIPIO,True
2,T15ymas,True
3,PEA,True
4,OCUPADOS,True
5,FORMALES,True
6,INFORMALES,True
7,EstisiINF,True


Variables faltantes: []


### 2.7 Revisión del descriptor oficial de ILMM 2020

Se revisa el archivo de descripción incluido en el paquete oficial de ILMM para documentar las variables laborales utilizadas en el proyecto.

Se prestará especial atención a:

- población de 15 años y más;
- población económicamente activa;
- población ocupada;
- población formal;
- población informal;
- tasas estimadas publicadas por ILMM.

La revisión permitirá determinar el denominador utilizado en las tasas publicadas y justificar la construcción de la tasa de informalidad laboral empleada en el modelo econométrico.

In [8]:
with zipfile.ZipFile(archivo_ilmm, "r") as z:

    descriptor_bytes_ilmm = z.read(
        "descripcion_archivos_fd.xlsx"
    )

excel_descriptor_ilmm = pd.ExcelFile(
    BytesIO(descriptor_bytes_ilmm)
)

print("Hojas disponibles en el descriptor ILMM:")

for hoja in excel_descriptor_ilmm.sheet_names:
    print(hoja)

Hojas disponibles en el descriptor ILMM:
Índice
1 
2
3
4
5
6
7


In [9]:
################  2.8 Cargar completa la base de estimaciones

ilmm_raw = pd.read_excel(
    BytesIO(estimaciones_bytes),
    sheet_name="Sheet 1"
)

print("Dimensiones de la base ILMM:")
print(ilmm_raw.shape)

print("\nNúmero de columnas:")
print(ilmm_raw.shape[1])

display(ilmm_raw.head(10))

Dimensiones de la base ILMM:
(2458, 15)

Número de columnas:
15


,LLAVE,MUNICIPIO,T15ymas,EstisiPEA,RECMPEA,EstisiOCU,RECMOCU,EstisiINF,RECMINF,PEA,NO_PEA,OCUPADOS,DESOCUPADOS,FORMALES,INFORMALES
0,1001,Aguascalientes,706146.451442,0.609054,0.006203,0.591558,0.005591,0.251977,0.026122,430081.451542,276064.999900,417726.792991,12354.658551,239794.282877,177932.510114
1,1002,Asientos,35175.941192,0.594906,0.010524,0.586544,0.003395,0.426814,0.014916,20926.370626,14249.570566,20632.222968,294.147657,5618.626438,15013.596531
2,1003,Calvillo,41447.094488,0.586366,0.011668,0.570542,0.005821,0.358614,0.012411,24303.158228,17143.936260,23647.320806,655.837423,8783.802127,14863.518679
3,1004,Cosío,11793.687552,0.596951,0.013634,0.581910,0.007064,0.323247,0.016689,7040.251063,4753.436489,6862.859091,177.391972,3050.583901,3812.275190
4,1005,Jesús María,91295.008850,0.617918,0.010040,0.601883,0.005953,0.298674,0.013192,56412.804983,34882.203867,54948.951243,1463.853740,27681.534445,27267.416798
5,1006,Pabellón de Arteaga,33427.873425,0.603009,0.010009,0.600903,0.003805,0.346279,0.021519,20157.299652,13270.573772,20086.925736,70.373917,8511.566423,11575.359312
6,1007,Rincón de Romos,39439.048672,0.593462,0.009932,0.594742,0.003798,0.355238,0.023308,23405.589527,16033.459146,23405.589527,0.000000,9395.324975,14010.264552
7,1008,San José de Gracia,6507.328147,0.577289,0.012292,0.563358,0.006399,0.394814,0.015394,3756.609718,2750.718429,3665.957417,90.652302,1096.775639,2569.181777
8,1009,Tepezalá,15546.257951,0.593880,0.012624,0.579751,0.007052,0.377612,0.016401,9232.619302,6313.638648,9012.958356,219.660946,3142.508914,5870.449442
9,1010,El Llano,14587.250897,0.602631,0.010501,0.590712,0.004507,0.361336,0.013949,8790.727742,5796.523155,8616.867297,173.860445,3345.963775,5270.903522


In [10]:

#####################   normalizamos la llave municipal.

ilmm_raw["CVEGEO"] = (
    ilmm_raw["LLAVE"]
    .astype("string")
    .str.strip()
    .str.zfill(5)
)

display(
    ilmm_raw[
        [
            "LLAVE",
            "CVEGEO",
            "MUNICIPIO"
        ]
    ].head(10)
)

,LLAVE,CVEGEO,MUNICIPIO
0,1001,01001,Aguascalientes
1,1002,01002,Asientos
2,1003,01003,Calvillo
3,1004,01004,Cosío
4,1005,01005,Jesús María
5,1006,01006,Pabellón de Arteaga
6,1007,01007,Rincón de Romos
7,1008,01008,San José de Gracia
8,1009,01009,Tepezalá
9,1010,01010,El Llano


In [11]:
###############  2.9 Control básico de cobertura

print("CONTROL INICIAL — ILMM 2020")
print("=" * 50)

print(
    "Número de registros:",
    f"{len(ilmm_raw):,}"
)

print(
    "CVEGEO únicos:",
    f"{ilmm_raw['CVEGEO'].nunique():,}"
)

print(
    "CVEGEO duplicados:",
    ilmm_raw["CVEGEO"].duplicated().sum()
)

print(
    "LLAVE nulos:",
    ilmm_raw["LLAVE"].isna().sum()
)

print(
    "OCUPADOS nulos:",
    ilmm_raw["OCUPADOS"].isna().sum()
)

print(
    "INFORMALES nulos:",
    ilmm_raw["INFORMALES"].isna().sum()
)

CONTROL INICIAL — ILMM 2020
Número de registros: 2,458
CVEGEO únicos: 2,458
CVEGEO duplicados: 0
LLAVE nulos: 0
OCUPADOS nulos: 0
INFORMALES nulos: 0


In [12]:
################2.10 Validaciones de coherencia laboral

ilmm_raw["check_ocupados"] = np.isclose(
    ilmm_raw["OCUPADOS"],
    ilmm_raw["FORMALES"] + ilmm_raw["INFORMALES"],
    rtol=1e-9,
    atol=1e-6
)

print(
    "Registros donde OCUPADOS != FORMALES + INFORMALES:",
    (~ilmm_raw["check_ocupados"]).sum()
)

Registros donde OCUPADOS != FORMALES + INFORMALES: 0


In [13]:
###  PEA = ocupados + desocupados
ilmm_raw["check_pea"] = np.isclose(
    ilmm_raw["PEA"],
    ilmm_raw["OCUPADOS"] + ilmm_raw["DESOCUPADOS"],
    rtol=1e-9,
    atol=1e-6
)

print(
    "Registros donde PEA != OCUPADOS + DESOCUPADOS:",
    (~ilmm_raw["check_pea"]).sum()
)

Registros donde PEA != OCUPADOS + DESOCUPADOS: 0


In [14]:
#################  2.11 Descubrir exactamente qué representan EstisiPEA, EstisiOCU y EstisiINF

ilmm_raw["PEA_T15_CALC"] = (
    ilmm_raw["PEA"]
    / ilmm_raw["T15ymas"]
)

ilmm_raw["OCU_T15_CALC"] = (
    ilmm_raw["OCUPADOS"]
    / ilmm_raw["T15ymas"]
)

ilmm_raw["INF_T15_CALC"] = (
    ilmm_raw["INFORMALES"]
    / ilmm_raw["T15ymas"]
)

comparacion_tasas_ilmm = ilmm_raw[
    [
        "CVEGEO",
        "MUNICIPIO",
        "EstisiPEA",
        "PEA_T15_CALC",
        "EstisiOCU",
        "OCU_T15_CALC",
        "EstisiINF",
        "INF_T15_CALC"
    ]
].copy()

display(comparacion_tasas_ilmm.head(10))

,CVEGEO,MUNICIPIO,EstisiPEA,PEA_T15_CALC,EstisiOCU,OCU_T15_CALC,EstisiINF,INF_T15_CALC
0,01001,Aguascalientes,0.609054,0.609054,0.591558,0.591558,0.251977,0.251977
1,01002,Asientos,0.594906,0.594906,0.586544,0.586544,0.426814,0.426814
2,01003,Calvillo,0.586366,0.586366,0.570542,0.570542,0.358614,0.358614
3,01004,Cosío,0.596951,0.596951,0.581910,0.581910,0.323247,0.323247
4,01005,Jesús María,0.617918,0.617918,0.601883,0.601883,0.298674,0.298674
5,01006,Pabellón de Arteaga,0.603009,0.603009,0.600903,0.600903,0.346279,0.346279
6,01007,Rincón de Romos,0.593462,0.593462,0.594742,0.593462,0.355238,0.355238
7,01008,San José de Gracia,0.577289,0.577289,0.563358,0.563358,0.394814,0.394814
8,01009,Tepezalá,0.593880,0.593880,0.579751,0.579751,0.377612,0.377612
9,01010,El Llano,0.602631,0.602631,0.590712,0.590712,0.361336,0.361336


In [15]:
# cuantificamos las diferencias

print(
    "Máxima diferencia EstisiPEA:",
    (
        ilmm_raw["EstisiPEA"]
        - ilmm_raw["PEA_T15_CALC"]
    ).abs().max()
)

print(
    "Máxima diferencia EstisiOCU:",
    (
        ilmm_raw["EstisiOCU"]
        - ilmm_raw["OCU_T15_CALC"]
    ).abs().max()
)

print(
    "Máxima diferencia EstisiINF:",
    (
        ilmm_raw["EstisiINF"]
        - ilmm_raw["INF_T15_CALC"]
    ).abs().max()
)

Máxima diferencia EstisiPEA: 1.1102230246251565e-16
Máxima diferencia EstisiOCU: 0.055662498336096045
Máxima diferencia EstisiINF: 0.3701495512708689


In [16]:
####  2.12 Construcción preliminar de nuestra tasa
ilmm_raw["TASA_INFORMALIDAD_2020"] = (
    ilmm_raw["INFORMALES"]
    / ilmm_raw["OCUPADOS"]
    * 100
)

In [17]:
# Control
print(
    "OCUPADOS <= 0:",
    (ilmm_raw["OCUPADOS"] <= 0).sum()
)

print(
    "Tasa de informalidad nula:",
    ilmm_raw["TASA_INFORMALIDAD_2020"]
    .isna()
    .sum()
)

print(
    "Tasa mínima:",
    ilmm_raw["TASA_INFORMALIDAD_2020"].min()
)

print(
    "Tasa máxima:",
    ilmm_raw["TASA_INFORMALIDAD_2020"].max()
)

OCUPADOS <= 0: 0
Tasa de informalidad nula: 0
Tasa mínima: 27.584043284024624
Tasa máxima: 100.0


### 2.13 Diccionario técnico de variables ILMM 2020

El archivo oficial de descripción de ILMM incluye una hoja específica para la base de estimaciones municipales utilizada en el proyecto.

Se documentan las variables necesarias para identificar cada municipio y construir la tasa de informalidad laboral.

La tasa utilizada en el proyecto se calculará a partir de las cifras finales de población ocupada e informal:

TASA_INFORMALIDAD_2020 = INFORMALES / OCUPADOS × 100

Esta definición permite medir qué proporción de la población ocupada estimada corresponde a ocupación informal.

In [18]:
descriptor_ilmm = pd.read_excel(
    BytesIO(descriptor_bytes_ilmm),
    sheet_name="6",
    header=2
)

# Limpiar encabezados
descriptor_ilmm.columns = (
    descriptor_ilmm.columns
    .astype(str)
    .str.strip()
)

print("Dimensiones del descriptor:")
print(descriptor_ilmm.shape)

print("\nColumnas:")
print(descriptor_ilmm.columns.tolist())

display(descriptor_ilmm)

Dimensiones del descriptor:
(15, 5)

Columnas:
['#', 'Nemónico', 'Nombre de la variable', 'Descripción', 'Fuente']


,#,Nemónico,Nombre de la variable,Descripción,Fuente
0,1,LLAVE,Código compuesto del municipio,Llave de 5 dígitos para identificar al municip...,NaN
1,2,MUNICIPIO,Nombre del municipio,Nombre del municipio,"ENOE, I T 2020"
2,3,T15ymas,Población de 15 años y más,Población de 15 años y más,Indicadores Laborales para los Municipios de M...
3,4,EstisiPEA,Estimación de la tasa de la población económic...,Estimación de la tasa de la población económic...,Indicadores Laborales para los Municipios de M...
4,5,RECMPEA,Raíz del error cuadrático medio de la tasa de ...,Raíz del error cuadrático de la estimación de ...,Indicadores Laborales para los Municipios de M...
5,6,EstisiOCU,Estimación de la tasa de la población ocupada,Estimación de la tasa de la población ocupada ...,Indicadores Laborales para los Municipios de M...
6,7,RECMOCU,Raíz del error cuadrático medio de la tasa de ...,Raíz del error cuadrático medio de la tasa de ...,Indicadores Laborales para los Municipios de M...
7,8,EstisiINF,Estimación de la tasa de la población ocupada ...,Estimación de la tasa de la población ocupada ...,Indicadores Laborales para los Municipios de M...
8,9,RECMINF,Raíz del error cuadrático medio de la tasa de ...,Raíz del error cuadrático medio de la tasa de ...,Indicadores Laborales para los Municipios de M...
9,10,PEA,Población económicamente activa de estimación ...,Cifra de la población económicamente activa de...,Indicadores Laborales para los Municipios de M...


In [19]:
## 2.14 Diccionario técnico de nuestras variables

variables_ilmm_interes = [
    "LLAVE",
    "MUNICIPIO",
    "T15ymas",
    "PEA",
    "OCUPADOS",
    "FORMALES",
    "INFORMALES",
    "EstisiINF"
]

diccionario_tecnico_ilmm = (
    descriptor_ilmm[
        descriptor_ilmm["Nemónico"]
        .isin(variables_ilmm_interes)
    ]
    .copy()
    .reset_index(drop=True)
)

display(diccionario_tecnico_ilmm)

print(
    "Variables encontradas:",
    diccionario_tecnico_ilmm["Nemónico"].nunique()
)

,#,Nemónico,Nombre de la variable,Descripción,Fuente
0,1,LLAVE,Código compuesto del municipio,Llave de 5 dígitos para identificar al municip...,NaN
1,2,MUNICIPIO,Nombre del municipio,Nombre del municipio,"ENOE, I T 2020"
2,3,T15ymas,Población de 15 años y más,Población de 15 años y más,Indicadores Laborales para los Municipios de M...
3,8,EstisiINF,Estimación de la tasa de la población ocupada ...,Estimación de la tasa de la población ocupada ...,Indicadores Laborales para los Municipios de M...
4,10,PEA,Población económicamente activa de estimación ...,Cifra de la población económicamente activa de...,Indicadores Laborales para los Municipios de M...
5,12,OCUPADOS,Población ocupada de estimación en áreas pequeñas,Cifra de la población ocupada derivada de esti...,Indicadores Laborales para los Municipios de M...
6,14,FORMALES,Población formal de estimación en áreas pequeñas,Cifra de la población formal derivada de estim...,Indicadores Laborales para los Municipios de M...
7,15,INFORMALES,Población informal de estimación en áreas pequ...,Cifra de la población informal derivada de est...,Indicadores Laborales para los Municipios de M...


Variables encontradas: 8


In [20]:
# 2.15 Documentar los valores en el límite de 100%

municipios_informalidad_100 = ilmm_raw[
    np.isclose(
        ilmm_raw["TASA_INFORMALIDAD_2020"],
        100.0
    )
][
    [
        "CVEGEO",
        "MUNICIPIO",
        "OCUPADOS",
        "INFORMALES",
        "TASA_INFORMALIDAD_2020"
    ]
].copy()

print(
    "Municipios con tasa de informalidad = 100%:",
    f"{len(municipios_informalidad_100):,}"
)

display(
    municipios_informalidad_100.head(20)
)

Municipios con tasa de informalidad = 100%: 188


,CVEGEO,MUNICIPIO,OCUPADOS,INFORMALES,TASA_INFORMALIDAD_2020
89,07010,Bejucal de Ocampo,2686.537532,2686.537532,100.0
90,07011,Bella Vista,7383.403755,7383.403755,100.0
101,07022,Chalchihuitán,7060.964069,7060.964069,100.0
102,07023,Chamula,34236.833553,34236.833553,100.0
103,07024,Chanal,3937.147873,3937.147873,100.0
112,07033,Francisco León,2724.656006,2724.656006,100.0
121,07042,Ixhuatán,4081.087630,4081.087630,100.0
135,07056,Mitontic,4218.977785,4218.977785,100.0
137,07058,Nicolás Ruíz,1835.275832,1835.275832,100.0
139,07060,Ocotepec,4499.796533,4499.796533,100.0


In [21]:
#3  para ocupación

ilmm_raw["TASA_OCUPACION_FINAL"] = (
    ilmm_raw["OCUPADOS"]
    / ilmm_raw["PEA"]
    * 100
)

municipios_ocupacion_100 = (
    np.isclose(
        ilmm_raw["TASA_OCUPACION_FINAL"],
        100.0
    )
).sum()

print(
    "Municipios con tasa de ocupación = 100%:",
    municipios_ocupacion_100
)

Municipios con tasa de ocupación = 100%: 67


### 2.16 Conclusión de Data Understanding — ILMM 2020

La revisión de ILMM 2020 permitió confirmar la calidad y consistencia de la información laboral municipal utilizada en el proyecto.

Principales resultados:

- La base contiene 2,458 municipios.
- Las 2,458 claves municipales son únicas.
- No existen claves duplicadas.
- No se encontraron valores faltantes en `OCUPADOS` ni `INFORMALES`.
- En todos los registros se cumple que `OCUPADOS = FORMALES + INFORMALES`.
- En todos los registros se cumple que `PEA = OCUPADOS + DESOCUPADOS`.
- La tasa de informalidad construida no presenta valores faltantes.
- La tasa de informalidad se encuentra entre aproximadamente 27.58% y 100%.
- Existen municipios cuya estimación calibrada implica que la totalidad de la población ocupada se clasifica como informal.
- Las variables `Estisi*` corresponden a estimaciones utilizadas en el procedimiento de estimación y calibración, por lo que no se utilizarán directamente como sustituto de la tasa final construida con los conteos calibrados.

La variable laboral del proyecto se define como:

`TASA_INFORMALIDAD_2020 = INFORMALES / OCUPADOS × 100`

Esta variable será utilizada como X3 en el modelo econométrico.



# Fase 3 — Data Preparation
## Construcción de la base municipal ILMM 2020

Se seleccionan únicamente las variables necesarias para el análisis econométrico.

La base resultante contendrá una observación por municipio y conservará las cifras de población ocupada e informal utilizadas para construir y auditar la tasa de informalidad.

La variable `TASA_INFORMALIDAD_2020` será utilizada como X3.

In [22]:
ilmm_2020_municipal = (
    ilmm_raw[
        [
            "CVEGEO",
            "MUNICIPIO",
            "OCUPADOS",
            "INFORMALES",
            "TASA_INFORMALIDAD_2020"
        ]
    ]
    .copy()
    .rename(
        columns={
            "MUNICIPIO": "municipio",
            "OCUPADOS": "OCUPADOS_2020",
            "INFORMALES": "INFORMALES_2020"
        }
    )
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Dimensiones de la base municipal ILMM:"
)

print(ilmm_2020_municipal.shape)

display(ilmm_2020_municipal.head(10))

Dimensiones de la base municipal ILMM:
(2458, 5)


,CVEGEO,municipio,OCUPADOS_2020,INFORMALES_2020,TASA_INFORMALIDAD_2020
0,01001,Aguascalientes,417726.792991,177932.510114,42.595427
1,01002,Asientos,20632.222968,15013.596531,72.767712
2,01003,Calvillo,23647.320806,14863.518679,62.854980
3,01004,Cosío,6862.859091,3812.275190,55.549373
4,01005,Jesús María,54948.951243,27267.416798,49.623180
5,01006,Pabellón de Arteaga,20086.925736,11575.359312,57.626336
6,01007,Rincón de Romos,23405.589527,14010.264552,59.858627
7,01008,San José de Gracia,3665.957417,2569.181777,70.082150
8,01009,Tepezalá,9012.958356,5870.449442,65.133436
9,01010,El Llano,8616.867297,5270.903522,61.169603


In [23]:
#  Control final

print("CONTROL DE CALIDAD — ILMM MUNICIPAL 2020")
print("=" * 55)

print(
    "Número de municipios:",
    f"{len(ilmm_2020_municipal):,}"
)

print(
    "CVEGEO únicos:",
    f"{ilmm_2020_municipal['CVEGEO'].nunique():,}"
)

print(
    "CVEGEO duplicados:",
    ilmm_2020_municipal["CVEGEO"]
    .duplicated()
    .sum()
)

print(
    "Valores nulos:",
    ilmm_2020_municipal
    .isna()
    .sum()
    .sum()
)

print(
    "Tasa < 0:",
    (
        ilmm_2020_municipal["TASA_INFORMALIDAD_2020"]
        < 0
    ).sum()
)

print(
    "Tasa > 100:",
    (
        ilmm_2020_municipal["TASA_INFORMALIDAD_2020"]
        > 100
    ).sum()
)

CONTROL DE CALIDAD — ILMM MUNICIPAL 2020
Número de municipios: 2,458
CVEGEO únicos: 2,458
CVEGEO duplicados: 0
Valores nulos: 0
Tasa < 0: 0
Tasa > 100: 0


In [24]:
assert (
    ilmm_2020_municipal["CVEGEO"].nunique()
    == len(ilmm_2020_municipal)
)

assert (
    ilmm_2020_municipal["CVEGEO"]
    .duplicated()
    .sum()
    == 0
)

assert (
    ilmm_2020_municipal
    .isna()
    .sum()
    .sum()
    == 0
)

assert (
    ilmm_2020_municipal["TASA_INFORMALIDAD_2020"]
    .between(0, 100)
    .all()
)

print(
    "Todas las validaciones fueron superadas correctamente."
)


Todas las validaciones fueron superadas correctamente.


### 3.3 Exportación de la base municipal ILMM 2020

Después de superar los controles de calidad, la base municipal ILMM 2020 se guarda como archivo procesado.

La base contiene una observación por municipio y conserva:

- `CVEGEO`
- municipio
- población ocupada estimada
- población informal estimada
- tasa de informalidad laboral 2020

La variable `TASA_INFORMALIDAD_2020` será utilizada posteriormente como la variable explicativa X3 del modelo econométrico.

In [25]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

archivo_salida_ilmm = (
    PROCESSED_DIR / "ilmm_2020_municipal.csv"
)

ilmm_2020_municipal.to_csv(
    archivo_salida_ilmm,
    index=False,
    encoding="utf-8-sig"
)

print("Base ILMM 2020 guardada correctamente.")
print(f"Ruta: {archivo_salida_ilmm}")

Base ILMM 2020 guardada correctamente.
Ruta: c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\ilmm_2020_municipal.csv


In [26]:
ilmm_verificacion = pd.read_csv(
    archivo_salida_ilmm,
    dtype={"CVEGEO": "string"}
)

print("Dimensiones:")
print(ilmm_verificacion.shape)

print(
    "CVEGEO únicos:",
    ilmm_verificacion["CVEGEO"].nunique()
)

print(
    "Valores nulos:",
    ilmm_verificacion.isna().sum().sum()
)

print(
    "Tasa mínima:",
    ilmm_verificacion["TASA_INFORMALIDAD_2020"].min()
)

print(
    "Tasa máxima:",
    ilmm_verificacion["TASA_INFORMALIDAD_2020"].max()
)

display(ilmm_verificacion.head(10))

Dimensiones:
(2458, 5)
CVEGEO únicos: 2458
Valores nulos: 0
Tasa mínima: 27.584043284024624
Tasa máxima: 100.0


,CVEGEO,municipio,OCUPADOS_2020,INFORMALES_2020,TASA_INFORMALIDAD_2020
0,01001,Aguascalientes,417726.792991,177932.510114,42.595427
1,01002,Asientos,20632.222968,15013.596531,72.767712
2,01003,Calvillo,23647.320806,14863.518679,62.854980
3,01004,Cosío,6862.859091,3812.275190,55.549373
4,01005,Jesús María,54948.951243,27267.416798,49.623180
5,01006,Pabellón de Arteaga,20086.925736,11575.359312,57.626336
6,01007,Rincón de Romos,23405.589527,14010.264552,59.858627
7,01008,San José de Gracia,3665.957417,2569.181777,70.082150
8,01009,Tepezalá,9012.958356,5870.449442,65.133436
9,01010,El Llano,8616.867297,5270.903522,61.169603


### 3.4 Auditoría de cobertura ILMM vs. muestra candidata

Se compara la cobertura municipal de ILMM 2020 con la muestra candidata construida previamente mediante DENUE, CONAPO y Censo 2020.

El objetivo es identificar:

- municipios presentes en todas las fuentes incorporadas hasta el momento;
- municipios candidatos sin información ILMM;
- municipios ILMM que no pertenecen a la muestra candidata.

La muestra continuará siendo provisional hasta incorporar CONEVAL 2020.

In [27]:
archivo_muestra_candidata = (
    PROCESSED_DIR
    / "muestra_candidata_denue_conapo_censo.csv"
)

muestra_candidata = pd.read_csv(
    archivo_muestra_candidata,
    dtype={"CVEGEO": "string"}
)

print(
    "Municipios candidatos antes de ILMM:",
    f"{len(muestra_candidata):,}"
)

print(
    "Municipios ILMM:",
    f"{len(ilmm_2020_municipal):,}"
)

Municipios candidatos antes de ILMM: 2,465
Municipios ILMM: 2,458


In [28]:
##Compara las claves

claves_candidatas = set(
    muestra_candidata["CVEGEO"]
)

claves_ilmm = set(
    ilmm_2020_municipal["CVEGEO"]
)

claves_comunes_ilmm = (
    claves_candidatas
    & claves_ilmm
)

solo_candidata = (
    claves_candidatas
    - claves_ilmm
)

solo_ilmm = (
    claves_ilmm
    - claves_candidatas
)

print(
    "Municipios comunes:",
    f"{len(claves_comunes_ilmm):,}"
)

print(
    "Candidatos sin ILMM:",
    f"{len(solo_candidata):,}"
)

print(
    "ILMM fuera de muestra candidata:",
    f"{len(solo_ilmm):,}"
)

Municipios comunes: 2,458
Candidatos sin ILMM: 7
ILMM fuera de muestra candidata: 0


In [29]:
### Identificar exactamente los municipios faltantes

detalle_candidatos_sin_ilmm = (
    muestra_candidata[
        muestra_candidata["CVEGEO"]
        .isin(solo_candidata)
    ]
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Candidatos sin correspondencia en ILMM:",
    len(detalle_candidatos_sin_ilmm)
)

display(detalle_candidatos_sin_ilmm)

Candidatos sin correspondencia en ILMM: 7


,CVEGEO,DENUE_2020,DENUE_2025,CONAPO
0,07120,True,True,True
1,07121,True,True,True
2,07122,True,True,True
3,07123,True,True,True
4,07124,True,True,True
5,17034,True,True,True
6,17035,True,True,True


In [30]:
detalle_ilmm_fuera_candidata = (
    ilmm_2020_municipal[
        ilmm_2020_municipal["CVEGEO"]
        .isin(solo_ilmm)
    ]
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Municipios ILMM fuera de muestra candidata:",
    len(detalle_ilmm_fuera_candidata)
)

display(detalle_ilmm_fuera_candidata)

Municipios ILMM fuera de muestra candidata: 0


,CVEGEO,municipio,OCUPADOS_2020,INFORMALES_2020,TASA_INFORMALIDAD_2020


### 3.5 Actualización de la muestra candidata después de ILMM 2020

La comparación de cobertura entre ILMM 2020 y la muestra candidata construida previamente permitió identificar 2,458 municipios con información disponible en todas las fuentes incorporadas hasta este momento.

Resultados:

- La muestra candidata antes de ILMM contenía 2,465 municipios.
- ILMM contiene información para 2,458 municipios.
- Los 2,458 municipios de ILMM están incluidos en la muestra candidata previa.
- Siete municipios de la muestra candidata no tienen correspondencia en ILMM.
- No se imputarán artificialmente tasas de informalidad para los municipios sin información laboral.

Por tanto, la muestra candidata se reduce de 2,465 a 2,458 municipios.

Esta muestra todavía no se considera definitiva, ya que falta incorporar la información socioeconómica de CONEVAL 2020.

3.6 Construir la nueva muestra candidata

In [31]:
muestra_candidata_ilmm = (
    muestra_candidata[
        muestra_candidata["CVEGEO"]
        .isin(claves_comunes_ilmm)
    ]
    .copy()
    .sort_values("CVEGEO")
    .reset_index(drop=True)
)

print(
    "Municipios en muestra candidata después de ILMM:",
    f"{len(muestra_candidata_ilmm):,}"
)

print(
    "CVEGEO únicos:",
    f"{muestra_candidata_ilmm['CVEGEO'].nunique():,}"
)

print(
    "CVEGEO duplicados:",
    muestra_candidata_ilmm["CVEGEO"]
    .duplicated()
    .sum()
)

Municipios en muestra candidata después de ILMM: 2,458
CVEGEO únicos: 2,458
CVEGEO duplicados: 0


In [32]:
# 3.7 Validar que X3 exista para los 2,458 municipios

validacion_x3 = (
    muestra_candidata_ilmm[["CVEGEO"]]
    .merge(
        ilmm_2020_municipal[
            [
                "CVEGEO",
                "TASA_INFORMALIDAD_2020"
            ]
        ],
        on="CVEGEO",
        how="left",
        validate="one_to_one"
    )
)

print(
    "Municipios candidatos:",
    f"{len(validacion_x3):,}"
)

print(
    "TASA_INFORMALIDAD_2020 faltantes:",
    validacion_x3[
        "TASA_INFORMALIDAD_2020"
    ].isna().sum()
)

print(
    "Tasa mínima:",
    validacion_x3[
        "TASA_INFORMALIDAD_2020"
    ].min()
)

print(
    "Tasa máxima:",
    validacion_x3[
        "TASA_INFORMALIDAD_2020"
    ].max()
)

Municipios candidatos: 2,458
TASA_INFORMALIDAD_2020 faltantes: 0
Tasa mínima: 27.584043284024624
Tasa máxima: 100.0


In [33]:
##### 3.8 Guardar la nueva muestra candidata

archivo_muestra_candidata_ilmm = (
    PROCESSED_DIR
    / "muestra_candidata_denue_conapo_censo_ilmm.csv"
)

muestra_candidata_ilmm.to_csv(
    archivo_muestra_candidata_ilmm,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Muestra candidata DENUE-CONAPO-Censo-ILMM "
    "guardada correctamente."
)

print(archivo_muestra_candidata_ilmm)

Muestra candidata DENUE-CONAPO-Censo-ILMM guardada correctamente.
c:\Users\nashe\Desktop\Econometría\PROYECTO\atractividad_comercial_Mexico\data\processed\muestra_candidata_denue_conapo_censo_ilmm.csv


### Conclusión de la preparación de ILMM 2020

La base municipal ILMM 2020 quedó construida y validada correctamente.

Principales resultados:

- ILMM contiene 2,458 municipios.
- Las 2,458 claves `CVEGEO` son únicas.
- No existen claves duplicadas.
- No se encontraron valores faltantes en la base procesada.
- Se comprobó que `OCUPADOS = FORMALES + INFORMALES`.
- Se comprobó que `PEA = OCUPADOS + DESOCUPADOS`.
- La tasa de informalidad se construyó como `INFORMALES / OCUPADOS × 100`.
- La tasa construida se encuentra dentro del intervalo válido de 0% a 100%.
- La muestra candidata previa contenía 2,465 municipios.
- Siete municipios candidatos no cuentan con información ILMM.
- La muestra candidata se reduce a 2,458 municipios.
- Los 2,458 municipios restantes cuentan con `TASA_INFORMALIDAD_2020`.

Por tanto, `TASA_INFORMALIDAD_2020` queda disponible como la variable explicativa laboral X3 del modelo econométrico.

La muestra sigue siendo provisional hasta incorporar CONEVAL 2020.